# Session 3, Module 06: Generators and Iterators


This module covers:
- Understanding iterators and the iterator protocol
- Creating generators with yield
- Generator expressions
- itertools for advanced iteration
- Memory-efficient data processing

Data Engineering Context:
Generators enable processing large datasets without loading
everything into memory — essential for ETL pipelines.


In [ ]:
import itertools
from typing import Iterator, Generator
import sys

## Iterators — The Protocol


In [ ]:
print("=== Iterators — The Protocol ===")

An iterator is an object that:
1. Has __iter__() method (returns itself)
2. Has __next__() method (returns next value or raises StopIteration)
Lists are iterable (have __iter__) but not iterators themselves

In [ ]:
my_list = [1, 2, 3]
print(f"List has __iter__: {hasattr(my_list, '__iter__')}")  # OUTPUT: True
print(f"List has __next__: {hasattr(my_list, '__next__')}")  # OUTPUT: False

# Get an iterator from the list
my_iter = iter(my_list)
print(f"Iterator has __next__: {hasattr(my_iter, '__next__')}")  # OUTPUT: True

# Manually iterate
print(f"\nManual iteration:")
print(f"  next(): {next(my_iter)}")  # OUTPUT: 1
print(f"  next(): {next(my_iter)}")  # OUTPUT: 2
print(f"  next(): {next(my_iter)}")  # OUTPUT: 3
# next(my_iter) would raise StopIteration

## Creating A Custom Iterator


In [ ]:
print("\n=== Creating a Custom Iterator ===")


class CountUp:
    """Iterator that counts from start to end."""

    def __init__(self, start: int, end: int):
        self.current = start
        self.end = end

    def __iter__(self):
        return self

    def __next__(self):
        if self.current > self.end:
            raise StopIteration
        value = self.current
        self.current += 1
        return value


# Usage
print("CountUp iterator:")
for num in CountUp(1, 5):
    print(f"  {num}")

## Generators — Simplified Iterators


In [ ]:
print("\n=== Generators — Simplified Iterators ===")

Generator functions use 'yield' instead of 'return'
They pause execution and resume on next()

In [ ]:
def count_up(start: int, end: int) -> Generator[int, None, None]:
    """Generator that counts from start to end."""
    current = start
    while current <= end:
        yield current  # Pause here, return value
        current += 1   # Resume here on next call


# Usage — exactly like the iterator class
print("count_up generator:")
for num in count_up(1, 5):
    print(f"  {num}")

# Generator state is preserved between yields
def show_state():
    """Demonstrate state preservation."""
    print("  Before first yield")
    yield 1
    print("  Between yields")
    yield 2
    print("  After last yield")


print("\nGenerator state demonstration:")
gen = show_state()
print(f"First: {next(gen)}")
print(f"Second: {next(gen)}")

## Generators Vs Lists — Memory Efficiency


In [ ]:
print("\n=== Generators vs Lists — Memory Efficiency ===")


def generate_squares(n: int) -> Generator[int, None, None]:
    """Generate squares lazily."""
    for i in range(n):
        yield i ** 2


def list_squares(n: int) -> list[int]:
    """Create list of squares eagerly."""
    return [i ** 2 for i in range(n)]


# Compare memory usage
n = 1000000

# Generator uses almost no memory
gen = generate_squares(n)
gen_size = sys.getsizeof(gen)
print(f"Generator size: {gen_size} bytes")

List uses significant memory
lst = list_squares(n)  # Don't run - uses ~8MB
Instead, estimate:

In [ ]:
list_size_estimate = n * 8  # Approximately 8 bytes per int
print(f"List size (estimated): {list_size_estimate:,} bytes ({list_size_estimate / (1024*1024):.1f} MB)")

# Generators compute values on-demand (lazy evaluation)
print(f"\nFirst 5 squares (lazy): {[next(gen) for _ in range(5)]}")

## Generator Expressions


In [ ]:
print("\n=== Generator Expressions ===")

Generator expressions are like list comprehensions but lazy
Use () instead of []
List comprehension — creates list in memory

In [ ]:
squares_list = [x ** 2 for x in range(10)]
print(f"List comprehension: {squares_list}")
print(f"  Type: {type(squares_list)}")

# Generator expression — creates generator
squares_gen = (x ** 2 for x in range(10))
print(f"Generator expression: {squares_gen}")
print(f"  Type: {type(squares_gen)}")
print(f"  Values: {list(squares_gen)}")

# Practical use: pass directly to functions
print(f"\nSum with generator: {sum(x ** 2 for x in range(10))}")  # No extra []

## Yield From — Delegating To Sub-Generators


In [ ]:
print("\n=== Yield From — Delegating to Sub-Generators ===")


def flatten(nested_list):
    """Flatten a nested list using yield from."""
    for item in nested_list:
        if isinstance(item, list):
            yield from flatten(item)  # Delegate to recursive call
        else:
            yield item


nested = [1, [2, 3, [4, 5]], 6, [7, 8]]
print(f"Nested: {nested}")
print(f"Flattened: {list(flatten(nested))}")


# Combining generators
def gen_a():
    yield 1
    yield 2


def gen_b():
    yield 3
    yield 4


def combined():
    yield from gen_a()
    yield from gen_b()


print(f"\nCombined generators: {list(combined())}")  # OUTPUT: [1, 2, 3, 4]

## Itertools — Powerful Iteration Tools


In [ ]:
print("\n=== Itertools — Powerful Iteration Tools ===")

itertools provides memory-efficient iteration functions
count — infinite counter

In [ ]:
print("itertools.count(10):")
counter = itertools.count(10)
print(f"  {next(counter)}, {next(counter)}, {next(counter)}")  # OUTPUT: 10, 11, 12

# cycle — infinite cycling
print("\nitertools.cycle(['A', 'B']):")
cycler = itertools.cycle(['A', 'B'])
print(f"  {[next(cycler) for _ in range(5)]}")  # OUTPUT: ['A', 'B', 'A', 'B', 'A']

# repeat — repeat value
print("\nitertools.repeat('X', 3):")
print(f"  {list(itertools.repeat('X', 3))}")  # OUTPUT: ['X', 'X', 'X']

# chain — combine iterables
print("\nitertools.chain([1,2], [3,4], [5,6]):")
print(f"  {list(itertools.chain([1, 2], [3, 4], [5, 6]))}")  # OUTPUT: [1, 2, 3, 4, 5, 6]

# islice — slice an iterator
print("\nitertools.islice(range(100), 5, 10):")
print(f"  {list(itertools.islice(range(100), 5, 10))}")  # OUTPUT: [5, 6, 7, 8, 9]

# takewhile — take while condition is true
print("\nitertools.takewhile(lambda x: x < 5, range(10)):")
print(f"  {list(itertools.takewhile(lambda x: x < 5, range(10)))}")  # OUTPUT: [0, 1, 2, 3, 4]

# dropwhile — drop while condition is true
print("\nitertools.dropwhile(lambda x: x < 5, range(10)):")
print(f"  {list(itertools.dropwhile(lambda x: x < 5, range(10)))}")  # OUTPUT: [5, 6, 7, 8, 9]

# groupby — group consecutive elements
print("\nitertools.groupby([1,1,2,2,2,3,1,1]):")
data = [1, 1, 2, 2, 2, 3, 1, 1]
for key, group in itertools.groupby(data):
    print(f"  {key}: {list(group)}")

## Practical Example: Processing Large Files


In [ ]:
print("\n=== Practical Example: Processing Large Files ===")


def read_lines_generator(filepath: str) -> Generator[str, None, None]:
    """
    Read file line by line using a generator.

    Memory efficient for large files.
    """
    with open(filepath, 'r') as f:
        for line in f:
            yield line.strip()


def process_records(records: Iterator[dict]) -> Generator[dict, None, None]:
    """
    Process records one at a time using a generator.

    Args:
        records: Iterator of raw records

    Yields:
        Processed records
    """
    for record in records:
        # Transform record
        processed = {
            **record,
            'processed': True,
            'name_upper': record.get('name', '').upper(),
        }
        yield processed


def batch_records(records: Iterator[dict], batch_size: int) -> Generator[list[dict], None, None]:
    """
    Batch records into groups.

    Args:
        records: Iterator of records
        batch_size: Number of records per batch

    Yields:
        Lists of records (batches)
    """
    batch = []
    for record in records:
        batch.append(record)
        if len(batch) >= batch_size:
            yield batch
            batch = []

    # Yield remaining records
    if batch:
        yield batch


# Demonstrate batching
sample_records = [{'id': i, 'name': f'Customer {i}'} for i in range(10)]

print("Batching records (batch_size=3):")
for i, batch in enumerate(batch_records(iter(sample_records), 3)):
    print(f"  Batch {i + 1}: {len(batch)} records")

## Practical Example: Data Pipeline


In [ ]:
print("\n=== Practical Example: Data Pipeline ===")


def extract_records(source: list[dict]) -> Generator[dict, None, None]:
    """Extract records from source."""
    for record in source:
        print(f"  Extract: {record['id']}")
        yield record


def transform_records(records: Iterator[dict]) -> Generator[dict, None, None]:
    """Transform records."""
    for record in records:
        transformed = {**record, 'status': 'transformed'}
        print(f"  Transform: {record['id']}")
        yield transformed


def load_records(records: Iterator[dict]) -> int:
    """Load records to destination."""
    count = 0
    for record in records:
        print(f"  Load: {record['id']}")
        count += 1
    return count


# Build pipeline with generators — nothing executes yet!
source_data = [{'id': i} for i in range(3)]

print("Building pipeline (no execution yet)...")
extracted = extract_records(source_data)
transformed = transform_records(extracted)

# Execute pipeline — pull-based, lazy evaluation
print("\nExecuting pipeline (pulling data)...")
loaded_count = load_records(transformed)
print(f"Total loaded: {loaded_count}")

## Generator Methods: Send(), Throw(), Close()


In [ ]:
print("\n=== Generator Methods: send(), throw(), close() ===")


def accumulator():
    """Generator that accepts values via send()."""
    total = 0
    while True:
        value = yield total
        if value is not None:
            total += value


# Using send() to pass values into generator
acc = accumulator()
print(f"Start: {next(acc)}")      # Must call next() first
print(f"Send 10: {acc.send(10)}")  # OUTPUT: 10
print(f"Send 5: {acc.send(5)}")    # OUTPUT: 15
print(f"Send 3: {acc.send(3)}")    # OUTPUT: 18

# close() stops the generator
acc.close()

## Summary


In [ ]:
print("\n=== Summary ===")
print("""
Generators and Iterators Key Points:

ITERATORS:
  - Objects with __iter__() and __next__()
  - iter() gets iterator from iterable
  - next() gets next value

GENERATORS:
  - Functions that use yield
  - Pause and resume execution
  - Memory efficient (lazy evaluation)

GENERATOR EXPRESSIONS:
  - (expr for x in iterable if condition)
  - Like list comprehensions but lazy

ITERTOOLS (common functions):
  - count(), cycle(), repeat(): Infinite iterators
  - chain(): Combine iterables
  - islice(): Slice iterators
  - takewhile(), dropwhile(): Conditional iteration
  - groupby(): Group consecutive elements
  - batched(): Create batches (Python 3.12+)

USE CASES:
  - Processing large files line by line
  - Streaming data pipelines
  - Infinite sequences
  - Memory-constrained environments
""")